# Restitution de graphe orienté avec networkx

Ce notebook construit un graphe orienté avec **networkx** à partir d'un
fichier texte **uploadé** (format `NOEUDS`/`ARETES`, identique au reste du
projet — voir `exemple.txt` ou `exemple_50_noeuds.txt` à la racine du
dépôt), puis le restitue avec **plotly** — zoomable, déplaçable, et chaque
nœud est cliquable (surligne prédécesseurs/successeurs directs) et
survolable (détail).

Il n'y a pas de graphe par défaut : tant qu'aucun fichier n'est uploadé
(section 2), rien ne s'affiche.

matplotlib et seaborn ont été écartés ici : dans un notebook, ils ne
produisent qu'une **image statique** (aucune interaction possible dessus).
matplotlib reste la bonne option pour un rendu non-interactif
(`src/restitution/`, `python-script/`) ou pour du clic *hors* notebook
(`python-script/` ouvre une vraie fenêtre graphique interactive).

⚠️ L'upload passe par `ipywidgets` et **nécessite un noyau Python actif**
(Jupyter, DataLab...). Dans l'export statique `index.html` (voir plus
bas), le bouton s'affiche mais l'upload ne déclenche rien : pour
l'utiliser, ouvrir le notebook lui-même dans Jupyter/VS Code.

## 1. Fonctions (parsing, layout, rendu)

Parsing du format `NOEUDS`/`ARETES`, détection de cycle, layout
hiérarchique et rendu plotly interactif — rien n'est exécuté ici, ces
fonctions sont appelées section 2 une fois un fichier uploadé.

In [2]:
# --- Imports : bibliothèques utilisées dans ce notebook -------------------
from collections import defaultdict  # dict qui crée une valeur par défaut au lieu de lever une erreur
from pathlib import Path              # manipulation de chemins de fichiers
import json                           # pour transformer des données Python en texte JSON (utilisé dans le JS ci-dessous)

import networkx as nx                 # bibliothèque de graphes : on l'utilise pour stocker et parcourir le graphe
import plotly.graph_objects as go     # pour dessiner le graphe de façon interactive (zoom, clic, survol)
from IPython.display import HTML, display  # pour afficher du HTML/JS personnalisé dans une cellule
import ipywidgets as widgets          # pour le bouton d'upload de fichier (utilisé section 2)

# En-têtes de section reconnus dans le fichier .txt (avec ou sans accent)
NODE_HEADERS = {"noeuds", "nœuds", "nodes"}
EDGE_HEADERS = {"aretes", "arêtes", "edges"}


def parse_graph(text: str) -> nx.DiGraph:
    """Parse le format NOEUDS/ARETES et construit un graphe orienté networkx."""
    # nx.DiGraph = graphe ORIENTÉ (Directed Graph) : chaque arête a un sens (A -> B ≠ B -> A)
    graph = nx.DiGraph()
    section = None  # section courante pendant qu'on lit le fichier ligne par ligne : None, "nodes" ou "edges"

    # .splitlines() découpe le texte en une liste de lignes (une string par ligne, sans les retours à la ligne)
    for raw_line in text.splitlines():
        line = raw_line.strip()  # .strip() enlève les espaces/tabulations en début et fin de ligne
        if not line or line.startswith("#"):
            continue  # ligne vide ou commentaire : on l'ignore et on passe à la suivante

        header = line.lower()  # .lower() : tout en minuscules, pour comparer sans se soucier de la casse
        if header in NODE_HEADERS:
            section = "nodes"
            continue
        if header in EDGE_HEADERS:
            section = "edges"
            continue

        if section == "nodes":
            graph.add_node(line)  # la ligne entière est le nom du nœud, ex: "A"
        elif section == "edges":
            parts = line.split()  # .split() sans argument : découpe sur les espaces, ex: "A B" -> ["A", "B"]
            if len(parts) < 2:
                raise ValueError(f"Ligne d'arête invalide (attendu 'source cible'): {raw_line!r}")
            graph.add_edge(parts[0], parts[1])  # crée une arête orientée parts[0] -> parts[1]
        else:
            # on a une ligne de contenu avant d'avoir vu "NOEUDS" ou "ARETES" : fichier mal formé
            raise ValueError(f"Ligne hors section (attendu NOEUDS/ARETES avant tout contenu): {raw_line!r}")

    if graph.number_of_nodes() == 0:
        raise ValueError("Aucun noeud trouvé dans le fichier.")

    return graph


def find_back_edges(graph: nx.DiGraph) -> set[tuple[str, str]]:
    """DFS classique : une arête vers un nœud déjà sur la pile d'appel referme un cycle."""
    # DFS = Depth-First Search = parcours en profondeur : on avance le plus loin possible
    # avant de revenir en arrière, comme si on explorait un labyrinthe couloir par couloir.
    back_edges: set[tuple[str, str]] = set()  # les arêtes qui "reviennent en arrière" -> créent un cycle
    visited: set[str] = set()   # tous les nœuds déjà visités, une fois pour toutes
    on_stack: set[str] = set()  # les nœuds actuellement "en cours d'exploration" (sur la pile d'appel)

    def dfs(node: str) -> None:
        visited.add(node)
        on_stack.add(node)
        for succ in graph.successors(node):  # .successors(node) = tous les nœuds pointés par node
            if succ in on_stack:
                # succ est un ancêtre de node dans l'exploration en cours : on referme un cycle
                back_edges.add((node, succ))
            elif succ not in visited:
                dfs(succ)  # appel récursif : la fonction s'appelle elle-même sur le nœud suivant
        on_stack.discard(node)  # on a fini d'explorer node, on le retire de la pile en cours

    for node in graph.nodes:
        if node not in visited:
            dfs(node)

    return back_edges


def layered_positions(graph: nx.DiGraph) -> dict[str, tuple[float, float]]:
    """Layout hiérarchique par niveaux (façon Sugiyama) : les arêtes de retour
    (cycles) sont exclues du calcul des niveaux via nx.topological_sort, mais
    restent dans le graphe dessiné."""
    # Idée générale : on place chaque nœud sur une "ligne" (son niveau), en fonction
    # de la distance depuis les nœuds sans prédécesseur, pour obtenir un dessin lisible
    # (comme un organigramme, du haut vers le bas).
    back_edges = find_back_edges(graph)

    # On construit une copie du graphe SANS les arêtes de cycle (back_edges) : un cycle
    # empêcherait de calculer un ordre topologique (impossible de dire qui vient "avant" qui).
    dag = nx.DiGraph()  # DAG = Directed Acyclic Graph = graphe orienté SANS cycle
    dag.add_nodes_from(graph.nodes)
    dag.add_edges_from(e for e in graph.edges if e not in back_edges)

    layer: dict[str, int] = {}  # niveau (0, 1, 2, ...) de chaque nœud
    # nx.topological_sort : range les nœuds dans un ordre où chaque nœud vient après ses prédécesseurs
    for node in nx.topological_sort(dag):
        preds = list(dag.predecessors(node))  # tous les nœuds qui pointent vers node
        # sans prédécesseur -> niveau 0 ; sinon -> 1 + le niveau le plus profond des prédécesseurs
        layer[node] = 0 if not preds else 1 + max(layer[p] for p in preds)

    # On regroupe les nœuds par niveau, pour ensuite les répartir horizontalement
    nodes_by_layer: dict[int, list[str]] = defaultdict(list)
    for node, lvl in layer.items():
        nodes_by_layer[lvl].append(node)

    pos: dict[str, tuple[float, float]] = {}  # position finale (x, y) de chaque nœud
    for lvl, nodes in nodes_by_layer.items():
        nodes.sort()  # ordre alphabétique à l'intérieur d'un même niveau, pour un rendu stable
        for i, node in enumerate(nodes):
            # x : centré autour de 0, réparti selon le nombre de nœuds du niveau
            # y : -lvl, pour que le niveau 0 soit en haut et les niveaux suivants en dessous
            pos[node] = (i - (len(nodes) - 1) / 2, -lvl)

    return pos


# Couleurs utilisées pour les nœuds selon leur état (normal, sélectionné, etc.)
_DEFAULT_COLOR = "#4C72B0"
_SELECTED_COLOR = "#f59e0b"
_PREDECESSOR_COLOR = "#0ea5e9"
_SUCCESSOR_COLOR = "#16a34a"
_DIMMED_COLOR = "#cbd5e1"

# Deux figures sur la même page embarquant chacune leur propre copie de
# plotly.js se marchent dessus (la 2e écrase "window.Plotly" au moment où
# la 1re en a encore besoin pour ses clics, vérifié en isolant le cas) :
# on n'embarque la bibliothèque qu'une seule fois, au premier appel.
_plotlyjs_embedded = {"done": False}


def render_directed_graph(graph, pos, div_id, node_size=36, show_labels=True,
                           arrow_size=1.4, arrow_width=1.5, standoff=None, height=550):
    """Figure plotly interactive : clique un nœud pour surligner ses
    prédécesseurs/successeurs directs, survole pour le détail.

    Le clic est géré en JavaScript pur (Plotly.restyle + l'événement natif
    "plotly_click"), pas via ipywidgets/ FigureWidget : ça fonctionne aussi
    bien dans un notebook avec noyau actif que dans un export HTML statique
    (nbconvert) ou sur GitHub, où aucun noyau Python n'est disponible pour
    répondre à un clic.
    """
    node_ids = list(graph.nodes)
    # "standoff" = distance entre le centre du nœud et la pointe de la flèche, pour que
    # la flèche ne rentre pas dans le cercle du nœud (moitié de sa taille + une petite marge)
    standoff = standoff if standoff is not None else node_size / 2 + 2

    # Texte affiché au survol de chaque nœud (liste dans le même ordre que node_ids)
    hover_text = [
        f"<b>{n}</b><br>prédécesseurs: {sorted(graph.predecessors(n)) or '—'}"
        f"<br>successeurs: {sorted(graph.successors(n)) or '—'}"
        for n in node_ids
    ]

    # go.Scatter = un ensemble de points sur un graphique (ici : un point par nœud)
    node_trace = go.Scatter(
        x=[pos[n][0] for n in node_ids], y=[pos[n][1] for n in node_ids],
        mode="markers+text" if show_labels else "markers",  # "markers+text" = points + leur nom affiché dessus
        text=node_ids if show_labels else None, textposition="middle center",
        textfont=dict(color="white", size=11),
        hovertext=hover_text, hoverinfo="text",
        marker=dict(size=node_size, color=[_DEFAULT_COLOR] * len(node_ids), line=dict(width=1.5, color="white")),
    )

    fig = go.Figure(
        data=[node_trace],
        layout=go.Layout(
            title=dict(text=f"plotly — {graph.number_of_nodes()} nœuds, {graph.number_of_edges()} arêtes "
                            "(clique un nœud pour ses connexions directes, survole pour le détail)"),
            showlegend=False, hovermode="closest",
            xaxis=dict(showgrid=False, zeroline=False, visible=False),
            yaxis=dict(showgrid=False, zeroline=False, visible=False),
            margin=dict(l=10, r=10, t=40, b=10), height=height,
        ),
    )

    # Les arêtes ne sont pas des "traits" classiques mais des annotations fléchées :
    # un Scatter en mode "lines" ne trace qu'un trait, sans tête de flèche.
    for u, v in graph.edges():
        fig.add_annotation(
            x=pos[v][0], y=pos[v][1], ax=pos[u][0], ay=pos[u][1],  # (ax, ay) = départ, (x, y) = pointe de la flèche
            xref="x", yref="y", axref="x", ayref="y",
            showarrow=True, arrowhead=2, arrowsize=arrow_size, arrowwidth=arrow_width,
            arrowcolor="#94a3b8", standoff=standoff, startstandoff=standoff,
        )

    # Jupyter insère les sorties HTML des cellules dynamiquement (pas un vrai
    # parsing HTML statique), ce qui fait qu'un <script src="..."> externe
    # s'y charge de façon asynchrone (même sans l'attribut async) — l'ordre
    # d'exécution avec le <script> suivant n'est alors plus garanti, et
    # Plotly.newPlot() peut s'exécuter avant que "Plotly" soit défini
    # (vérifié en isolant le cas). Embarquer plotly.js directement (inline,
    # donc synchrone) évite le problème — mais une seule fois au total.
    include_js = not _plotlyjs_embedded["done"]
    # fig.to_html(...) transforme la figure Python en un bloc HTML/JS prêt à afficher
    plot_html = fig.to_html(full_html=False, include_plotlyjs=include_js, div_id=div_id)
    _plotlyjs_embedded["done"] = True
    # Pour chaque nœud, la liste de ses prédécesseurs/successeurs, à disposition du JavaScript
    # ci-dessous (le clic est géré côté navigateur, pas en rappelant du code Python)
    predecessors_map = {n: sorted(graph.predecessors(n)) for n in node_ids}
    successors_map = {n: sorted(graph.successors(n)) for n in node_ids}

    # Le bloc ci-dessous est du JavaScript (pas du Python) : json.dumps(...) sert à
    # convertir les listes/dicts Python en syntaxe JavaScript valide (même écriture que JSON).
    click_script = f"""
    <script>
    (function() {{
      var nodeIds = {json.dumps(node_ids)};
      var predecessors = {json.dumps(predecessors_map)};
      var successors = {json.dumps(successors_map)};
      var DEFAULT = "{_DEFAULT_COLOR}", SELECTED = "{_SELECTED_COLOR}",
          PRED = "{_PREDECESSOR_COLOR}", SUCC = "{_SUCCESSOR_COLOR}", DIM = "{_DIMMED_COLOR}";
      var selected = null;

      function attach() {{
        var plotDiv = document.getElementById("{div_id}");
        if (!plotDiv || !plotDiv.on) {{ setTimeout(attach, 100); return; }}

        plotDiv.on("plotly_click", function(data) {{
          var idx = data.points[0].pointIndex;
          var clicked = nodeIds[idx];
          var colors;
          if (selected === clicked) {{
            selected = null;
            colors = nodeIds.map(function() {{ return DEFAULT; }});
          }} else {{
            selected = clicked;
            var preds = predecessors[clicked] || [];
            var succs = successors[clicked] || [];
            colors = nodeIds.map(function(n) {{
              if (n === clicked) return SELECTED;
              if (preds.indexOf(n) !== -1) return PRED;
              if (succs.indexOf(n) !== -1) return SUCC;
              return DIM;
            }});
          }}
          Plotly.restyle(plotDiv, {{ "marker.color": [colors] }}, [0]);
        }});
      }}
      attach();
    }})();
    </script>
    """

    return HTML(plot_html + click_script)


## 2. Upload d'un fichier `.txt`

Choisis un fichier au format `NOEUDS`/`ARETES` depuis ton poste (par
exemple `exemple.txt` ou `exemple_50_noeuds.txt` à la racine du dépôt) :
les statistiques et le graphe interactif s'affichent automatiquement en
dessous.

In [3]:
# Crée le bouton d'upload : accepte uniquement les fichiers .txt, un seul à la fois
upload_widget = widgets.FileUpload(accept=".txt", multiple=False, description="Choisir un fichier .txt")
# Output() = une zone de la page où on peut afficher du texte/graphique par programme
# (ce qu'on print()/display() dans le "with upload_output:" ci-dessous apparaît ici)
upload_output = widgets.Output()


def _handle_upload(change):
    """Appelée automatiquement dès qu'un fichier est choisi dans le bouton (voir
    upload_widget.observe(...) tout en bas)."""
    with upload_output:  # tout print()/display() fait ici s'affiche dans la zone upload_output
        upload_output.clear_output()  # efface l'affichage précédent avant d'afficher le nouveau graphe
        if not upload_widget.value:
            print("Aucun fichier.")
            return
        # upload_widget.value : tuple des fichiers choisis (un seul ici, donc [0])
        uploaded = upload_widget.value[0]  # ipywidgets>=8 : dict avec name/type/size/content/last_modified

        # try/except : si le fichier n'est pas au bon format (mauvais encodage, sections
        # NOEUDS/ARETES manquantes, ligne d'arête incomplète...), on affiche un message clair
        # au lieu du traceback Python complet — plus confortable pour un simple "mauvais fichier".
        try:
            # "content" est en bytes (contenu brut du fichier) : bytes(...) le convertit en bytes Python,
            # puis .decode("utf-8") le transforme en texte lisible
            text = bytes(uploaded["content"]).decode("utf-8")
            graph = parse_graph(text)             # construit le graphe networkx
            pos = layered_positions(graph)         # calcule où placer chaque nœud à l'écran
            cycle = len(find_back_edges(graph)) > 0  # True s'il y a au moins un cycle
        except UnicodeDecodeError:
            # levée par .decode("utf-8") si le fichier n'est pas du texte UTF-8 valide
            # (ex: fichier binaire, ou texte dans un autre encodage comme Windows-1252)
            print(f"Fichier invalide ({uploaded['name']}) : ce n'est pas un fichier texte UTF-8.")
            return
        except ValueError as exc:
            # levée par parse_graph (voir cellule précédente) : format NOEUDS/ARETES incorrect
            print(f"Fichier invalide ({uploaded['name']}) : {exc}")
            return

        print(f"{uploaded['name']} : {graph.number_of_nodes()} nœuds, {graph.number_of_edges()} arêtes, cycle détecté : {cycle}")
        print("Nœuds :", list(graph.nodes))
        print("Arêtes :", list(graph.edges))
        # display(...) affiche le graphe interactif juste en dessous des lignes de statistiques
        display(render_directed_graph(graph, pos, div_id="graph-plot-upload"))


# .observe(...) : à chaque fois que la valeur du bouton change (= un fichier est choisi),
# Python appelle automatiquement _handle_upload — pas besoin de cliquer sur un 2e bouton "valider"
upload_widget.observe(_handle_upload, names="value")
display(upload_widget, upload_output)


FileUpload(value=(), accept='.txt', description='Choisir un fichier .txt')

Output()